# PDF Extraction & Chunking Experiment — Technical Documentation


| Phase | Variable | Method |
|---|---|---|
| 1 | Extraction quality | Visual inspection — A vs B vs C |
| 2 | Chunking strategy | RAGAS — SW vs SB-v3 vs RC on Phase 1 winner |
| 3 | Contextual enrichment | RAGAS — with vs without header |

In [1]:
import sys, os, re, json, asyncio, unicodedata, warnings
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from typing import Literal
import numpy as np
import tiktoken
import pymupdf
import pdfplumber
import pymupdf4llm

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')

PDF_DIR  = ROOT / 'data' / 'raw' / 'pdfs'
EVAL_DIR = ROOT / 'eval' / 'golden_dataset' / 'pdf_technical'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

PDFS      = sorted(PDF_DIR.glob('*.pdf'))
MAX_PAGES = 20
_ENC      = tiktoken.encoding_for_model('gpt-4o')

print(f'Found {len(PDFS)} PDFs:')
for p in PDFS:
    doc = pymupdf.open(str(p))
    print(f'  {p.name:<45} {len(doc):>4} pages')
    doc.close()
print(f'\npymupdf: {pymupdf.__version__} | pymupdf4llm: {pymupdf4llm.__version__}')

Found 5 PDFs:
  FastAPI_CLI.pdf                                  4 pages
  Overview _ Kubernetes.pdf                        3 pages
  React Fundamentals.pdf                          14 pages
  Stripe API Reference.pdf                        17 pages
  aws-overview.pdf                               161 pages

pymupdf: 1.27.2.3 | pymupdf4llm: 1.27.2.3


In [3]:
def _token_count(text):
    return len(_ENC.encode(text))

def fix_hyphenation(text):
    return re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)

def normalize_ligatures(text):
    return unicodedata.normalize('NFKD', text)

def remove_headers_footers(pages):
    all_lines = []
    for page in pages:
        all_lines.extend(l.strip() for l in page.splitlines() if l.strip())
    counts = Counter(all_lines)
    threshold = max(2, len(pages) * 0.7)
    noise = {line for line, cnt in counts.items() if cnt >= threshold}
    return ['\n'.join(l for l in page.splitlines() if l.strip() not in noise) for page in pages]

def quality_gate(pages):
    return [{'page': i+1, 'chars': len(p.strip()),
             'quality': 'ok' if len(p.strip()) >= 50 else 'LOW'}
            for i, p in enumerate(pages)]

def clean_pipeline(pages):
    pages = [fix_hyphenation(p) for p in pages]
    pages = [normalize_ligatures(p) for p in pages]
    return remove_headers_footers(pages)

def find_near(text, keyword, window=700):
    idx = text.lower().find(keyword.lower())
    if idx == -1: return f"  ['{keyword}' not found]"
    return text[max(0, idx-40): idx+window]


In [6]:
def extract_a(pdf_path, max_pages=MAX_PAGES):
    doc = pymupdf.open(str(pdf_path))
    pages = []
    for i, page in enumerate(doc):
        if i >= max_pages: break
        pages.append(page.get_text())
    doc.close()
    return '\n\n---PAGE---\n\n'.join(clean_pipeline(pages))

print('Strategy A (raw get_text) defined.')

Strategy A (raw get_text) defined.


In [7]:
@dataclass
class ExtractedBlock:
    type: str
    content: str
    page: int
    section: str = ''
    heading_level: int = 0

def _body_font_size(page_dict):
    sizes = [round(span['size'],1)
             for block in page_dict.get('blocks',[])
             if block.get('type')==0
             for line in block.get('lines',[])
             for span in line.get('spans',[])]
    return Counter(sizes).most_common(1)[0][0] if sizes else 11.0

def _is_mono(font_name):
    return any(k in font_name.lower() for k in
               ('mono','courier','code','consol','inconsolata','fixed','typewriter','source code'))

def _overlaps(b1, b2):
    return not (b1[2]<b2[0] or b1[0]>b2[2] or b1[3]<b2[1] or b1[1]>b2[3])

def _text_blocks_b(page, page_num, table_bboxes):
    pd = page.get_text('dict')
    body = _body_font_size(pd)
    out  = []
    raw  = sorted([b for b in pd['blocks'] if b.get('type')==0],
                  key=lambda b: (round(b['bbox'][1]/20)*20, b['bbox'][0]))
    for block in raw:
        if any(_overlaps(block['bbox'], tb) for tb in table_bboxes): continue
        lines, is_mono, max_sz, is_bold = [], False, 0.0, False
        for line in block.get('lines',[]):
            spans = []
            for span in line.get('spans',[]):
                t = span['text'].strip()
                if not t: continue
                spans.append(t)
                if _is_mono(span.get('font','')): is_mono = True
                sz = round(span['size'],1)
                if sz > max_sz: max_sz = sz
                if 'bold' in span.get('font','').lower(): is_bold = True
            if spans: lines.append(' '.join(spans))
        content = '\n'.join(lines).strip()
        if not content: continue
        if is_mono:
            out.append(ExtractedBlock('code', content, page_num))
        elif max_sz >= body+5 or (max_sz >= body+3 and is_bold):
            out.append(ExtractedBlock('heading', content, page_num, heading_level=1))
        elif max_sz >= body+2.5 or (max_sz >= body+1 and is_bold):
            out.append(ExtractedBlock('heading', content, page_num, heading_level=2))
        elif max_sz >= body+0.5 and is_bold:
            out.append(ExtractedBlock('heading', content, page_num, heading_level=3))
        else:
            out.append(ExtractedBlock('paragraph', content, page_num))
    return out

def _tables_b(pdf_path, page_num):
    out = []
    try:
        with pdfplumber.open(str(pdf_path)) as pdf:
            if page_num >= len(pdf.pages): return []
            for table in pdf.pages[page_num].extract_tables():
                if not table or len(table)<2 or len(table[0])<2: continue
                headers = [str(h or '').strip() for h in table[0]]
                rows    = [[str(c or '').strip() for c in row] for row in table[1:]]
                md = '\n'.join(
                    ['| '+' | '.join(headers)+' |',
                     '| '+' | '.join(['---']*len(headers))+' |'] +
                    ['| '+' | '.join(row)+' |' for row in rows if any(row)]
                )
                out.append(ExtractedBlock('table', md, page_num+1))
    except Exception: pass
    return out

def extract_b(pdf_path, max_pages=MAX_PAGES):
    doc = pymupdf.open(str(pdf_path))
    all_blocks, section = [], ''
    for pg in range(min(max_pages, len(doc))):
        try: tbboxes = [t.bbox for t in doc[pg].find_tables().tables]
        except: tbboxes = []
        for b in _text_blocks_b(doc[pg], pg+1, tbboxes):
            if b.type == 'heading': section = b.content
            b.section = section
            all_blocks.append(b)
        for b in _tables_b(pdf_path, pg):
            b.section = section
            all_blocks.append(b)
    doc.close()

    text_pages = {}
    for b in all_blocks:
        if b.type != 'table': text_pages.setdefault(b.page, []).append(b)
    cleaned = clean_pipeline([('\n'.join(bb.content for bb in text_pages.get(i+1,[]))) for i in range(max_pages)])
    cleaned_map = {i+1: cleaned[i] for i in range(len(cleaned))}

    final, tables = [], [b for b in all_blocks if b.type == 'table']
    tp = {}
    for b in all_blocks:
        if b.type != 'table': tp.setdefault(b.page, []).append(b)
    for pg, ct in cleaned_map.items():
        for b in tp.get(pg, []):
            words = b.content.split()[:3]
            if b.content.strip() and any(len(w)>3 and w in ct for w in words):
                final.append(b)
    final.extend(tables)
    final.sort(key=lambda b: (b.page, b.type != 'heading'))
    return final

def blocks_to_text(blocks):
    parts = []
    for b in blocks:
        if b.type == 'heading':
            parts.append(('#'*b.heading_level if b.heading_level else '##') + ' ' + b.content)
        elif b.type == 'code': parts.append(f'```\n{b.content}\n```')
        else: parts.append(b.content)
    return '\n\n'.join(parts)

print('Strategy B (layout-aware + pdfplumber) defined.')

Strategy B (layout-aware + pdfplumber) defined.


In [8]:
def extract_c(pdf_path, max_pages=MAX_PAGES):
    doc = pymupdf.open(str(pdf_path))
    actual = min(max_pages, len(doc))
    doc.close()
    md = pymupdf4llm.to_markdown(str(pdf_path), pages=list(range(actual)), show_progress=False)
    pages = md.split('\n-----\n')
    return '\n\n'.join(clean_pipeline(pages))

print('Strategy C (pymupdf4llm) defined.')

Strategy C (pymupdf4llm) defined.


In [9]:
results_a, results_b, results_c = {}, {}, {}
for pdf in PDFS:
    name = pdf.name
    print(f'Processing: {name}')
    results_a[name] = extract_a(pdf)
    results_b[name] = extract_b(pdf)
    results_c[name] = extract_c(pdf)
    blocks = results_b[name]
    h = sum(1 for b in blocks if b.type=='heading')
    t = sum(1 for b in blocks if b.type=='table')
    c = sum(1 for b in blocks if b.type=='code')
    print(f'  A: {len(results_a[name].split()):>5} words')
    print(f'  B: {len(blocks):>3} blocks | {h} headings | {t} tables | {c} code')
    print(f'  C: {len(results_c[name].split()):>5} words\n')

Processing: FastAPI_CLI.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.



Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


  A:   582 words
  B:  67 blocks | 1 headings | 0 tables | 39 code
  C:   592 words

Processing: Overview _ Kubernetes.pdf


...

=== Document parser messages ===
                                                            Using Tesseract for OCR processing.

  A:  1444 words
  B:  66 blocks | 8 headings | 1 tables | 0 code
  C:  1488 words

Processing: React Fundamentals.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

=== Document parser messages ===
                                                                                                Using Tesseract for OCR processing.

  A:  1414 words
  B: 195 blocks | 8 headings | 0 tables | 0 code
  C:  1369 words

Processing: Stripe API Reference.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

=== Document parser messages ===
                                                                                                                                    Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=1/2.
OCR on page.number=4/5.
OCR on page.number=5/6.
OCR on page.number=6/7.
OCR on page.number=7/8.
OCR on page.number=9/10.
OCR on page.number=14/15.
OCR on page.number=15/16.

  A:  3404 words
  B: 140 blocks | 2 headings | 2 tables | 0 code
  C:  3294 words

Processing: aws-overview.pdf
=== Document parser messages ===
                                                                                                                                                                                                                                                                                                                                                                                                     Using Tesseract for OCR processing.
OCR on

- Strategy B's heading detection is unreliable across PDFs (Stripe: 2 headings, React: only 8 out of 195 blocks)
- Strategy C's word counts match A closely, meaning OCR recovery works well
- FastAPI's code block detection (39) is Strategy B's strongest showing

## Phase 1: Visual Inspection — A vs B vs C

In [10]:
TARGET = 'Stripe API Reference.pdf'
print(f'=== {TARGET} first 1200 chars ===\n')
print('── A ──'); print(results_a[TARGET][:1200])
print('\n── B ──'); print(blocks_to_text(results_b[TARGET])[:1200])
print('\n── C ──'); print(results_c[TARGET][:1200])

=== Stripe API Reference.pdf first 1200 chars ===

── A ──
API Reference
The Stripe API is organized around REST. Our API has predictable resource-oriented URLs, accepts
form-encoded request bodies, returns JSON-encoded responses, and uses standard HTTP response
codes, authentication, and verbs.
You can use the Stripe API in sandboxes without affecting your live data or interacting with banking
networks. The API key that you use to authenticate the request determines whether the request runs
in live mode or in a sandbox. Sandboxes support all v2 APIs. Test mode sandboxes support some
v2 APIs.
The Stripe API doesn’t support bulk updates. You can work on only one object per request.
The Stripe API differs for every account as we release new versions and tailor functionality. Log in to
see docs with your test key and data.
Just getting started?
Check out our development quickstart guide.
Not a developer?
Use Stripe’s no-code options or apps from our partners to get started with Stripe and

In [11]:
# Table quality — Stripe parameter tables
print('=== TABLE QUALITY — Stripe API Reference ===\n')
print('── A: around "parameter" ──'); print(find_near(results_a[TARGET], 'parameter'))
print('\n── B: tables extracted ──')
b_tables = [b for b in results_b[TARGET] if b.type=='table']
print(f'{len(b_tables)} tables detected')
for i, t in enumerate(b_tables[:3]):
    print(f'\n[Table {i+1}] p={t.page} section={t.section[:40]}')
    print(t.content[:500])
print('\n── C: around "parameter" ──'); print(find_near(results_c[TARGET], 'parameter'))

=== TABLE QUALITY — Stripe API Reference ===

── A: around "parameter" ──
 information provided (e.g., a required parameter was omitted, a charge failed, etc.).
Codes in the 5xx  range indicate an error with Stripe’s servers (these are rare).
Some 4xx  errors that could be handled programmatically (e.g., a card is declined) include an error
code that briefly explains the error reported.
YOUR API KEY
A sample test API key is included in all the examples here, so you can test any example right away.
Do not submit any personally identifiable information in requests made with this key.
To test requests using your account, replace the sample API key with your actual API key or
sign in.
AUTHENTICATED REQUEST
cURL
1
curl https://api.stripe.com/v1/charges \
2
  -u 
:
3
# The colon prevents curl from asking for 

── B: tables extracted ──
2 tables detected

[Table 1] p=5 section=E xp a n di n g R e spons e s
|  |  |  | Card errors are the most common type of error you should expect to
card_erro

In [12]:
# Code block quality — FastAPI CLI
TARGET_CODE = 'FastAPI_CLI.pdf'
print(f'=== CODE BLOCKS — {TARGET_CODE} ===\n')
print('── A ──'); print(find_near(results_a[TARGET_CODE], 'fastapi', 500))
print('\n── B code blocks ──')
cbs = [b for b in results_b[TARGET_CODE] if b.type=='code']
print(f'{len(cbs)} code blocks')
for i, cb in enumerate(cbs[:3]): print(f'\n[{i+1}]\n{cb.content[:250]}')
print('\n── C ──'); print(find_near(results_c[TARGET_CODE], 'fastapi', 500))

=== CODE BLOCKS — FastAPI_CLI.pdf ===

── A ──
FastAPI 
Learn 
FastAPI CLI
FastAPI CLI is a command line program that you can use to serve your FastAPI app, manage
your FastAPI project, and more.
When you install FastAPI (e.g. with pip install "fastapi[standard]" ), it comes with a
command line program you can run in the terminal.
To run your FastAPI app for development, you can use the fastapi dev  command:
fast →
$ fastapi dev
   FastAPI   Starting development server 🚀
             Searching for package file structure 
from directories wit

── B code blocks ──
39 code blocks

[1]
When you install FastAPI (e.g. with pip install "fastapi[standard]" ), it comes with a

[2]
To run your FastAPI app for development, you can use the fastapi dev command:

[3]
bash

── C ──


FastAPI Learn 

## FastAPI CLI 

FastAPI CLI is a command line program that you can use to serve your FastAPI app, manage your FastAPI project, and more. 

When you install FastAPI (e.g. with `pip install "fastapi[standa

In [13]:
# Heading detection
print('Headings detected (Strategy B):\n')
for name, blocks in results_b.items():
    headings = [b for b in blocks if b.type=='heading']
    print(f'── {name} ({len(headings)}) ──')
    for h in headings[:6]:
        print(f'  {"  "*(h.heading_level-1)}[H{h.heading_level}] p.{h.page} — {h.content[:60]}')
    if len(headings)>6: print(f'  ...and {len(headings)-6} more')
    print()

Headings detected (Strategy B):

── FastAPI_CLI.pdf (1) ──
  [H1] p.1 — FastAPI CLI

── Overview _ Kubernetes.pdf (8) ──
  [H1] p.1 — Overview
  [H1] p.1 — Overview
    [H2] p.1 — Kubernetes is a portable, extensible, open source platform f
  [H1] p.1 — Why you need Kubernetes and what it can do
  [H1] p.1 — What Kubernetes is not
  [H1] p.2 — Historical context for Kubernetes
  ...and 2 more

── React Fundamentals.pdf (8) ──
  [H1] p.1 — React Fundamentals
  [H1] p.1 — Your first component
  [H1] p.5 — Custom Components
    [H2] p.6 — INFO
  [H1] p.7 — Props
    [H2] p.9 — NOTE
  ...and 2 more

── Stripe API Reference.pdf (2) ──
  [H1] p.2 — E rrors
  [H1] p.7 — I de mpot e nt r e qu e sts

── aws-overview.pdf (18) ──
  [H1] p.2 — Overview of Amazon Web Services: AWS Whitepaper
  [H1] p.3 — Table of Contents
  [H1] p.11 — Overview of Amazon Web Services
  [H1] p.11 — Introduction
  [H1] p.12 — What is cloud computing?
  [H1] p.13 — Six advantages of cloud computing
  ...and 12 more



In [14]:
# Quality gate
print('Quality gate:\n')
for pdf in PDFS:
    doc = pymupdf.open(str(pdf))
    pages = [page.get_text() for i, page in enumerate(doc) if i < MAX_PAGES]
    doc.close()
    low = [r for r in quality_gate(pages) if r['quality']!='ok']
    print(f'  {pdf.name}: {"✅ OK" if not low else f"⚠️  {len(low)} low-quality pages"}')
    for r in low: print(f'    → Page {r["page"]}: {r["chars"]} chars')

Quality gate:

  FastAPI_CLI.pdf: ✅ OK
  Overview _ Kubernetes.pdf: ✅ OK
  React Fundamentals.pdf: ✅ OK
  Stripe API Reference.pdf: ✅ OK
  aws-overview.pdf: ✅ OK


In [ ]:
import pprint
phase1_assessment = {
    'Strategy A': {'code_blocks':'?', 'table_quality':'?', 'heading_structure':'?', 'noise':'?'},
    'Strategy B': {'code_blocks':'?', 'table_quality':'?', 'heading_structure':'?', 'noise':'?'},
    'Strategy C': {'code_blocks':'?', 'table_quality':'?', 'heading_structure':'?', 'noise':'?'},
    'winner': 'C',  # update after reviewing output above
}
pprint.pprint(phase1_assessment)
EXTRACTION_WINNER = phase1_assessment['winner']
print(f'\nPhase 2 will use extraction: {EXTRACTION_WINNER}')

## Findings — PDF Extraction Strategy Comparison

### Strategies Tested
- **A** — Raw text extraction (`pymupdf get_text()`)
- **B** — Layout-aware extraction (font-size heading detection + pdfplumber tables)
- **C** — pymupdf4llm (PDF → Markdown converter)

### PDFs Tested
5 technical documentation PDFs: FastAPI CLI, Kubernetes Overview, React Fundamentals,
Stripe API Reference, AWS Overview

---

### Strategy A — Raw Text
- Extracts words correctly across all 5 PDFs
- No structure preserved — headings, code blocks, and tables are all flat text
- Code examples are mixed inline with prose
- Acceptable baseline, not production-ready

### Strategy B — Layout-Aware
- **Breaks on font-encoded PDFs** — Stripe output showed character spacing artifacts
  (`n e twor k s`, `T he API ke y`) making text unreadable
- Misclassified prose sentences as code blocks on FastAPI (39 "code blocks" detected,
  most were actually inline-code prose)
- Heading detection inconsistent: only 2 headings on Stripe (both garbled),
  18 clean headings on AWS — performance depends entirely on font quality
- Too brittle for production use across diverse PDF sources

### Strategy C — pymupdf4llm
- Converts all PDFs to clean Markdown consistently
- Headings detected as `##` / `###` without any font-size tuning required
- Inline code correctly wrapped in backticks
- Tables rendered as Markdown tables
- Images skipped gracefully with a placeholder note
- Word counts match Strategy A — no content loss despite richer output
- Works on Stripe where Strategy B completely failed
- Falls back to Tesseract OCR automatically on image-heavy pages

---

**Winner: Strategy C (pymupdf4llm)**
Reason: Preserves document structure as Markdown, requires no font-size tuning,
and degrades gracefully on difficult PDFs where Strategy B completely breaks.



In [16]:
EXTRACTION_WINNER = 'C'

## Chunking Strategies — SW / SB-v3 / RC

In [20]:
from backend.models import Chunk, SourceType
from backend.connectors.chunkers.sliding_window_chunker import SlidingWindowChunker
sw_chunker = SlidingWindowChunker(window_tokens=512, overlap_tokens=50)

def _sliding_split(text, max_tokens, overlap_tokens):
    tokens = _ENC.encode(text)
    if len(tokens) <= max_tokens: return [text]
    step   = max_tokens - overlap_tokens
    return [_ENC.decode(tokens[s:s+max_tokens]) for s in range(0, len(tokens), step) if tokens[s:s+max_tokens]]

def _make_chunk(content, metadata, chunk_type):
    return Chunk(
        tenant_id=metadata.get('tenant_id',''),
        source_url=metadata.get('source_url',''),
        source_type=SourceType(metadata.get('source_type','pdf')),
        content=content,
        metadata={**metadata, 'chunk_type': chunk_type}
    )

# ── SemanticBlock v3: min_heading_words=2 for short tech headings ──
def semantic_block_chunk(blocks, metadata, max_tokens=512, overlap_tokens=50, min_heading_words=2):
    chunks, buf_texts, buf_tokens, section = [], [], 0, metadata.get('source_url','')

    def _flush():
        if buf_texts:
            chunks.append(_make_chunk('\n\n'.join(buf_texts), {**metadata,'section':section}, 'text'))
            buf_texts.clear()

    for block in blocks:
        content = block.content.strip()
        if not content: continue
        is_heading = (
            block.type == 'heading'
            and len(content.split()) >= min_heading_words
            and not re.match(r'^[\d\s\.\-]+$', content)
        )
        if is_heading:
            _flush()
            buf_texts[:] = [content]
            buf_tokens = _token_count(content)
            section    = content
        elif block.type in ('table','code'):
            _flush()
            chunks.append(_make_chunk(content, {**metadata,'section':section}, block.type))
        else:
            tok = _token_count(content)
            if tok > max_tokens:
                _flush()
                for part in _sliding_split(content, max_tokens, overlap_tokens):
                    chunks.append(_make_chunk(part, {**metadata,'section':section}, 'paragraph'))
            elif buf_tokens + tok > max_tokens:
                _flush(); buf_texts[:] = [content]; buf_tokens = tok
            else:
                buf_texts.append(content); buf_tokens += tok
    _flush()
    return chunks

# ── Recursive Character Chunker ──
def recursive_chunk(text, metadata, max_tokens=512, overlap_tokens=50):
    seps = ['\n\n','\n','. ',' ']
    def _split(text, si):
        if _token_count(text) <= max_tokens: return [text]
        if si >= len(seps): return _sliding_split(text, max_tokens, overlap_tokens)
        sep   = seps[si]
        parts = text.split(sep)
        result, cur = [], ''
        for part in parts:
            if not part.strip(): continue
            cand = (cur+sep+part) if cur else part
            if _token_count(cand) <= max_tokens:
                cur = cand
            else:
                if cur: result.append(cur.strip())
                if _token_count(part) > max_tokens: result.extend(_split(part, si+1)); cur=''
                else: cur = part
        if cur.strip(): result.append(cur.strip())
        return result
    raw = _split(text, 0)
    out = []
    for i, ct in enumerate(raw):
        if not ct.strip(): continue
        if i > 0 and overlap_tokens > 0:
            prev = _ENC.encode(raw[i-1])
            ct   = _ENC.decode(prev[-overlap_tokens:]) + ' ' + ct
        out.append(_make_chunk(ct.strip(), metadata, 'recursive'))
    return out

print('SlidingWindow, SemanticBlock-v3, RecursiveChar defined.')

SlidingWindow, SemanticBlock-v3, RecursiveChar defined.


In [21]:
chunks_sw, chunks_sb, chunks_rc = {}, {}, {}

for pdf in PDFS:
    name = pdf.name
    meta = {'tenant_id':'exp','source_url':name,'source_type':'pdf'}

    if EXTRACTION_WINNER == 'B':
        blocks = results_b[name]
        flat   = blocks_to_text(blocks)
        chunks_sw[name] = sw_chunker.chunk(flat, meta)
        chunks_sb[name] = semantic_block_chunk(blocks, meta)
        chunks_rc[name] = recursive_chunk(flat, meta)
    else:  # C
        text = results_c[name]
        md_blocks = []
        for line in text.split('\n'):
            s = line.strip()
            if not s: continue
            if s.startswith('### '): md_blocks.append(ExtractedBlock('heading',s[4:],1,heading_level=3))
            elif s.startswith('## '): md_blocks.append(ExtractedBlock('heading',s[3:],1,heading_level=2))
            elif s.startswith('# '): md_blocks.append(ExtractedBlock('heading',s[2:],1,heading_level=1))
            elif s.startswith('```'): md_blocks.append(ExtractedBlock('code',s,1))
            elif s.startswith('|'):
                if md_blocks and md_blocks[-1].type == 'table':
                    md_blocks[-1] = ExtractedBlock('table', md_blocks[-1].content + '\n' + s, 1)
                else:
                    md_blocks.append(ExtractedBlock('table', s, 1))

            else: md_blocks.append(ExtractedBlock('paragraph',s,1))
        chunks_sw[name] = sw_chunker.chunk(text, meta)
        chunks_sb[name] = semantic_block_chunk(md_blocks, meta)
        chunks_rc[name] = recursive_chunk(text, meta)

print(f'{"PDF":<45} {"SW":>5} {"SB":>5} {"RC":>5}')
print('-'*65)
for pdf in PDFS:
    n = pdf.name
    sb_types = Counter(c.metadata.get('chunk_type','?') for c in chunks_sb[n])
    print(f'{n:<45} {len(chunks_sw[n]):>5} {len(chunks_sb[n]):>5} {len(chunks_rc[n]):>5}   SB types: {dict(sb_types)}')

PDF                                              SW    SB    RC
-----------------------------------------------------------------
FastAPI_CLI.pdf                                   3     6     2   SB types: {'text': 6}
Overview _ Kubernetes.pdf                         5     8     4   SB types: {'text': 8}
React Fundamentals.pdf                            5     6     5   SB types: {'text': 6}
Stripe API Reference.pdf                         13    35    13   SB types: {'text': 34, 'table': 1}
aws-overview.pdf                                 12    21    14   SB types: {'text': 21}


In [23]:
# Inspect first 3 chunks of each strategy on Stripe
INSPECT = 'Stripe API Reference.pdf'
print(f'Chunk quality: {INSPECT}\n')
for label, cd in [('SlidingWindow',chunks_sw),('SemanticBlock-v3',chunks_sb),('Recursive',chunks_rc)]:
    print(f'── {label} ({len(cd[INSPECT])} chunks) ──')
    for i, c in enumerate(cd[INSPECT][:3]):
        ct = c.metadata.get('chunk_type','?')
        sec = c.metadata.get('section','')[:35]
        print(f'  [{i+1}] [{ct}] sec="{sec}" ({_token_count(c.content)} tok)')
        print(f'  {c.content[:200]}')
        print()
    print()

Chunk quality: Stripe API Reference.pdf

── SlidingWindow (13 chunks) ──
  [1] [?] sec="" (512 tok)
  

Sign in → 

INIntroTRODducUtionCTION 

## API Reference 

**==> picture [269 x 10] intentionally omitted <==**

The Stripe API is organized around REST. Our API has predictable resource-oriented URL

  [2] [?] sec="" (512 tok)
  ] intentionally omitted <==**

AUTHENTICATED REQUEST cURL<br>**----- End of picture text -----**<br>


`1 curl https://api.stripe.com/v1/charges \ 2 -u sk_test_tR3PYbcVNZZ796tH88S4VQ2sk_test_tR3PYbc..

  [3] [?] sec="" (512 tok)
  The type of error returned. One of `api_error` , `card_error` , `idempotency_error` , or `invalid_ request_error` 

Possible enum values 


card_error

idempotency_error
invalid_request_error

## More


── SemanticBlock-v3 (35 chunks) ──
  [1] [text] sec="Stripe API Reference.pdf" (12 tok)
  Sign in →

INIntroTRODducUtionCTION

  [2] [text] sec="API Reference" (168 tok)
  API Reference

**==> picture [269 x 10] intentionally omitted

## Eval Set Generation

In [24]:
from openai import AsyncOpenAI
client = AsyncOpenAI()

QA_PROMPT = """You are building an evaluation dataset for a technical documentation RAG system.
Generate exactly 8 question-answer pairs from the document excerpt.
Rules:
- 2 about specific API parameters, flags, or config values
- 2 about code examples or commands shown
- 2 about how a concept or feature works
- 2 that require understanding a full section
Respond as JSON: {"questions": [{"question":"...","answer":"...","type":"..."}]}
Answers must be answerable from the text only."""

async def gen_qa(text, pdf_name, n=8):
    resp = await client.chat.completions.create(
        model='gpt-4o-mini', temperature=0.3,
        response_format={'type':'json_object'},
        messages=[{'role':'system','content':QA_PROMPT},
                  {'role':'user','content':f'Document: {pdf_name}\n\n{text[:6000]}\n\nGenerate {n} QA pairs.'}]
    )
    parsed = json.loads(resp.choices[0].message.content)
    pairs  = next((v for v in parsed.values() if isinstance(v,list)), [])
    for p in pairs: p['source_pdf'] = pdf_name
    return pairs[:n]

all_qa = []
for pdf in PDFS:
    name  = pdf.name
    pairs = await gen_qa(results_c[name], name)
    all_qa.extend(pairs)
    print(f'✓ {name}: {len(pairs)} pairs')

print(f'\nTotal: {len(all_qa)} QA pairs')
qa_path = EVAL_DIR / 'pdf_technical_eval_v1.jsonl'
qa_path.write_text('\n'.join(json.dumps(q) for q in all_qa))
print(f'Saved → {qa_path}')

✓ FastAPI_CLI.pdf: 8 pairs
✓ Overview _ Kubernetes.pdf: 8 pairs
✓ React Fundamentals.pdf: 8 pairs
✓ Stripe API Reference.pdf: 8 pairs
✓ aws-overview.pdf: 8 pairs

Total: 40 QA pairs
Saved → /Users/mdayanarshad/Desktop/Switch Job UAE/kapa-inspired-rag-mcp/eval/golden_dataset/pdf_technical/pdf_technical_eval_v1.jsonl


In [25]:
for qa in all_qa[:6]:
    print(f'[{qa.get("type","?")}] {qa["question"]}')
    print(f'  → {qa["answer"][:120]}')
    print()

[specific API parameters] What command is used to run a FastAPI app in development mode?
  → The command used to run a FastAPI app in development mode is `fastapi dev`.

[specific API parameters] How can you configure the app entrypoint in the pyproject.toml file?
  → You can configure the app entrypoint in the pyproject.toml file by setting it as: entrypoint = 'main:app'.

[code examples or commands shown] What does the command 'fastapi dev main.py' do?
  → The command 'fastapi dev main.py' initiates development mode and guesses the FastAPI app object to use from the specifie

[code examples or commands shown] What is the purpose of the auto-reload feature in 'fastapi dev'?
  → The auto-reload feature in 'fastapi dev' automatically reloads the server when you make changes to your code.

[how a concept or feature works] How does FastAPI CLI determine which app to run by default?
  → FastAPI CLI tries to detect automatically the FastAPI app to run, assuming it's an object called 'app' i

## Phase 2: RAGAS — Chunking Strategies

In [26]:
from backend.strategies.embedding.openai_embedding import OpenAIEmbedding
embedder = OpenAIEmbedding()

async def embed_chunks(chunks):
    if not chunks: return chunks, np.empty((0,1536), dtype=np.float32)
    texts = [c.content for c in chunks]
    vecs  = []
    for i in range(0, len(texts), 100):
        vecs.extend(await embedder.embed(texts[i:i+100]))
    m = np.array(vecs, dtype=np.float32)
    return chunks, m / np.maximum(np.linalg.norm(m, axis=1, keepdims=True), 1e-9)

async def retrieve(query, chunks, matrix, top_k=5):
    q = np.array((await embedder.embed([query]))[0], dtype=np.float32)
    q = q / max(np.linalg.norm(q), 1e-9)
    return [chunks[i] for i in np.argsort(matrix @ q)[::-1][:top_k]]

async def answer(question, contexts):
    resp = await client.chat.completions.create(
        model='gpt-4o-mini', temperature=0,
        messages=[{'role':'system','content':'Answer using only the provided context. Be specific and concise.'},
                  {'role':'user','content':f'Context:\n{chr(10).join(contexts)}\n\nQuestion: {question}'}]
    )
    return resp.choices[0].message.content

async def run_eval(all_chunks, qa_pairs, label, enrich_fn=None):
    if not all_chunks: print(f'  [{label}] SKIPPED'); return []
    print(f'Embedding {len(all_chunks)} chunks [{label}]...')
    chunks, matrix = await embed_chunks(all_chunks)
    records = []
    for qa in qa_pairs:
        retrieved = await retrieve(qa['question'], chunks, matrix)
        contexts  = [enrich_fn(c) for c in retrieved] if enrich_fn else [c.content for c in retrieved]
        records.append({'question':qa['question'], 'answer': await answer(qa['question'], contexts),
                        'ground_truth':qa['answer'], 'contexts':contexts})
    print(f'  [{label}] done.')
    return records

print('RAG eval runner ready.')

RAG eval runner ready.


In [27]:
all_sw = [c for n in chunks_sw for c in chunks_sw[n]]
all_sb = [c for n in chunks_sb for c in chunks_sb[n]]
all_rc = [c for n in chunks_rc for c in chunks_rc[n]]

records_sw = await run_eval(all_sw, all_qa, 'SlidingWindow')
records_sb = await run_eval(all_sb, all_qa, 'SemanticBlock-v3')
records_rc = await run_eval(all_rc, all_qa, 'RecursiveChar')

Embedding 38 chunks [SlidingWindow]...
  [SlidingWindow] done.
Embedding 76 chunks [SemanticBlock-v3]...
  [SemanticBlock-v3] done.
Embedding 38 chunks [RecursiveChar]...
  [RecursiveChar] done.


In [28]:
from datasets import Dataset
from ragas import aevaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

jllm   = LangchainLLMWrapper(ChatOpenAI(model='gpt-4o-mini', temperature=0))
jembed = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model='text-embedding-3-small'))
mets   = [Faithfulness(), AnswerRelevancy(), ContextPrecision(), ContextRecall()]

def agg(sc):
    df = sc.to_pandas()
    return {c: round(df[c].mean(),4) for c in ['faithfulness','answer_relevancy','context_precision','context_recall'] if c in df}

print('Scoring SlidingWindow...')
res_sw = agg(await aevaluate(Dataset.from_list(records_sw), metrics=mets, llm=jllm, embeddings=jembed))
print('Scoring SemanticBlock-v3...')
res_sb = agg(await aevaluate(Dataset.from_list(records_sb), metrics=mets, llm=jllm, embeddings=jembed))
print('Scoring RecursiveChar...')
res_rc = agg(await aevaluate(Dataset.from_list(records_rc), metrics=mets, llm=jllm, embeddings=jembed))
print('Done.')

Scoring SlidingWindow...


Evaluating:   0%|          | 0/160 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Scoring SemanticBlock-v3...


Evaluating:   0%|          | 0/160 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Scoring RecursiveChar...


Evaluating:   0%|          | 0/160 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g

Done.


In [29]:
p2 = {'SlidingWindow':res_sw, 'SemanticBlock-v3':res_sb, 'RecursiveChar':res_rc}
print(f'{"Strategy":<22} {"Faith":>8} {"AnsRel":>8} {"CtxPrec":>8} {"CtxRec":>8} {"SUM":>8}')
print('-'*70)
best_sum, phase2_winner = 0, ''
for label, res in p2.items():
    f,r,p,c = res.get('faithfulness',0),res.get('answer_relevancy',0),res.get('context_precision',0),res.get('context_recall',0)
    s = f+r+p+c
    mark = ' ◄' if s > best_sum else ''
    if s > best_sum: best_sum, phase2_winner = s, label
    print(f'{label:<22} {f:>8.4f} {r:>8.4f} {p:>8.4f} {c:>8.4f} {s:>8.4f}{mark}')
print(f'\n✅ Phase 2 winner: {phase2_winner}')

Strategy                  Faith   AnsRel  CtxPrec   CtxRec      SUM
----------------------------------------------------------------------
SlidingWindow            0.8838   0.8205   0.7898   0.8750   3.3691 ◄
SemanticBlock-v3         0.8604   0.8346   0.7510   0.8167   3.2627
RecursiveChar            0.8879   0.8173   0.7947   0.9250   3.4249 ◄

✅ Phase 2 winner: RecursiveChar


## Phase 2 Findings — Chunking Strategy Comparison

### Setup
- Extraction: Strategy C (pymupdf4llm) — winner from Phase 1
- Chunkers tested: SlidingWindow, SemanticBlock-v3, RecursiveChar
- Eval set: 40 QA pairs across 5 technical documentation PDFs
- Retrieval: top-5 cosine similarity, GPT-4o-mini for answers

### Results

| Strategy        | Faithfulness | Ans Relevancy | Ctx Precision | Ctx Recall | SUM    |
|-----------------|-------------|---------------|---------------|------------|--------|
| SlidingWindow   | 0.8838      | 0.8205        | 0.7898        | 0.8750     | 3.3691 |
| SemanticBlock-v3| 0.8604      | 0.8346        | 0.7510        | 0.8167     | 3.2627 |
| RecursiveChar   | **0.8879**  | 0.8173        | **0.7947**    | **0.9250** | **3.4249** |

### What happened

**RecursiveChar** came out on top, mostly because of Context Recall (0.9250). 
It retrieved 92.5% of the information actually needed to answer the questions.
The reason is straightforward — it splits on natural text boundaries (`\n\n` first,
then `\n`, then `. `) instead of blindly chopping at a token count. 
Since pymupdf4llm already outputs clean Markdown paragraphs, those boundaries are 
already there. RC just uses them.

**SemanticBlock** lost despite being the most "structured" approach. The problem was 
the same as in the previous experiment — noise chunks. The Stripe PDF navigation 
sidebar produced tiny 12-token chunks like `"Sign in → INIntroTRODducUtionCTION"` 
and `"Just getting started?"`. These got retrieved and wasted context slots, dragging 
Context Precision down to 0.7510. The section-boundary splitting also scattered 
related information across different chunks, which hurt recall (0.8167).

**SlidingWindow** was consistent but hit a ceiling. Fixed token windows sometimes 
cut mid-sentence or right in the middle of a concept explanation. RC's fallback 
hierarchy avoids that.

### The real takeaway

SemanticBlock needs clean, well-structured input to work. When the extractor 
(Strategy B) was unreliable, block classification made things worse. But when 
the extractor (Strategy C) already gives you clean Markdown, you don't need 
semantic classification — RecursiveChar picks up the structure for free.

### Winner

**Strategy C (pymupdf4llm) + RecursiveChar**

## Phase 3: Contextual Enrichment

In [30]:
def enrich(chunk):
    meta = chunk.metadata or {}
    doc  = Path(chunk.source_url).stem if chunk.source_url else 'Unknown'
    sec  = meta.get('section','')[:80]
    ct   = meta.get('chunk_type','text')
    return f'Document: {doc}\nSection: {sec}\nType: {ct}\n\n{chunk.content}'

winner_map  = {'SlidingWindow':chunks_sw,'SemanticBlock-v3':chunks_sb,'RecursiveChar':chunks_rc}
winner_flat = [c for n in winner_map[phase2_winner] for c in winner_map[phase2_winner][n]]

records_enriched = await run_eval(winner_flat, all_qa, f'{phase2_winner}+Enrichment', enrich_fn=enrich)
res_enriched = agg(await aevaluate(Dataset.from_list(records_enriched), metrics=mets, llm=jllm, embeddings=jembed))

baseline = p2[phase2_winner]
print(f'\nPhase 3: Contextual Enrichment on {phase2_winner}')
print(f'{"Metric":<25} {"Without":>12} {"With":>12} {"Delta":>10}')
print('-'*63)
for m in ['faithfulness','answer_relevancy','context_precision','context_recall']:
    wo,wi = baseline.get(m,0), res_enriched.get(m,0)
    d = wi-wo
    print(f'{m:<25} {wo:>12.4f} {wi:>12.4f} {"▲" if d>0.005 else ("▼" if d<-0.005 else "~")}{abs(d):>8.4f}')
print(f'\n✅ Enrichment: {"ADOPT" if sum(res_enriched.values())>sum(baseline.values()) else "SKIP"}')

Embedding 38 chunks [RecursiveChar+Enrichment]...
  [RecursiveChar+Enrichment] done.


Evaluating:   0%|          | 0/160 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Phase 3: Contextual Enrichment on RecursiveChar
Metric                         Without         With      Delta
---------------------------------------------------------------
faithfulness                    0.8879       0.9151 ▲  0.0272
answer_relevancy                0.8173       0.8166 ~  0.0007
context_precision               0.7947       0.7970 ~  0.0023
context_recall                  0.9250       0.8917 ▼  0.0333

✅ Enrichment: SKIP


In [32]:
# Final comparison
final = {'SlidingWindow':res_sw, 'SemanticBlock-v3':res_sb, 'RecursiveChar':res_rc,
         f'{phase2_winner}+Enrichment':res_enriched}
print('='*90)
print('FINAL RESULTS')
print('='*90)
print(f'{"Strategy":<40} {"Faith":>8} {"AnsRel":>8} {"CtxPrec":>8} {"CtxRec":>8} {"SUM":>8}')
print('-'*90)
best, blabel = 0, ''
for label, res in final.items():
    f,r,p,c = res.get('faithfulness',0),res.get('answer_relevancy',0),res.get('context_precision',0),res.get('context_recall',0)
    s = f+r+p+c
    if s>best: best,blabel=s,label
    print(f'{label:<40} {f:>8.4f} {r:>8.4f} {p:>8.4f} {c:>8.4f} {s:>8.4f}')
print(f'\nWINNER: {blabel}')

FINAL RESULTS
Strategy                                    Faith   AnsRel  CtxPrec   CtxRec      SUM
------------------------------------------------------------------------------------------
SlidingWindow                              0.8838   0.8205   0.7898   0.8750   3.3691
SemanticBlock-v3                           0.8604   0.8346   0.7510   0.8167   3.2627
RecursiveChar                              0.8879   0.8173   0.7947   0.9250   3.4249
RecursiveChar+Enrichment                   0.9151   0.8166   0.7970   0.8917   3.4204

WINNER: RecursiveChar


## Experiment 06 — PDF Extraction & Chunking: Conclusions

### Setup
- **Corpus:** 5 technical documentation PDFs (FastAPI CLI, Kubernetes Overview,
  React Fundamentals, Stripe API Reference, AWS Overview)
- **Eval set:** 40 QA pairs generated by GPT-4o-mini, 8 per PDF,
  covering API parameters, code examples, concept explanations, and full-section questions
- **Metrics:** RAGAS — Faithfulness, Answer Relevancy, Context Precision, Context Recall

---

### Phase 1 — Extraction Strategy

Tested three strategies:
- **A:** Raw `get_text()` via PyMuPDF
- **B:** Layout-aware extraction with font-size heading detection + pdfplumber tables
- **C:** pymupdf4llm (PDF → Markdown converter)

**Winner: C (pymupdf4llm)**

Strategy B failed on the Stripe PDF due to non-standard font encoding, producing
character-spaced garbage (`n e twor k s`). It also misclassified prose sentences
containing inline code references as code blocks (FastAPI: 39 "code blocks",
most were actually prose). Strategy A extracted clean text but threw away all structure.

Strategy C produced consistent clean Markdown across all 5 PDFs — headings as `##`,
inline code as backticks, tables as Markdown tables, images gracefully skipped.
It fell back to Tesseract OCR automatically on image-heavy pages (Stripe: 9 pages,
AWS: 3 pages) without any configuration.

Quality gate: all 5 PDFs passed — no scanned-only pages detected.

---

### Phase 2 — Chunking Strategy

Tested three chunkers on Strategy C output:
- **SlidingWindow:** fixed 512-token windows with 50-token overlap
- **SemanticBlock-v3:** heading-aware splitting with `min_heading_words=2`
- **RecursiveChar:** hierarchical splitting (`\n\n` → `\n` → `. ` → ` `)

| Strategy         | Faithfulness | Ans Relevancy | Ctx Precision | Ctx Recall | SUM    |
|------------------|-------------|---------------|---------------|------------|--------|
| SlidingWindow    | 0.8838      | 0.8205        | 0.7898        | 0.8750     | 3.3691 |
| SemanticBlock-v3 | 0.8604      | 0.8346        | 0.7510        | 0.8167     | 3.2627 |
| RecursiveChar    | **0.8879**  | 0.8173        | **0.7947**    | **0.9250** | **3.4249** |

**Winner: RecursiveChar**

Context Recall of 0.9250 was the decisive metric — RC retrieved 92.5% of the
information needed to answer questions correctly. The reason is that pymupdf4llm
already produces clean Markdown with natural paragraph breaks. RecursiveChar
exploits those breaks without any block classification overhead.

SemanticBlock underperformed despite being the most structured approach.
The Stripe PDF's navigation sidebar produced 12-token noise chunks
(`"Sign in → INIntroTRODducUtionCTION"`, `"Just getting started?"`) that got
retrieved and wasted context slots, dragging Context Precision down to 0.7510.
Section-boundary splitting also scattered related information across chunks,
hurting recall (0.8167).

SlidingWindow was consistent but hit a ceiling — fixed token windows occasionally
cut mid-sentence or mid-concept.

---

### Phase 3 — Contextual Enrichment

Tested prepending a `Document / Section / Type` header to each chunk before embedding
(Anthropic Contextual Retrieval pattern) on the Phase 2 winner.

| Metric            | Without    | With       | Delta     |
|-------------------|------------|------------|-----------|
| Faithfulness      | 0.8879     | 0.9151     | ▲ +0.0272 |
| Answer Relevancy  | 0.8173     | 0.8166     | ~ neutral |
| Context Precision | 0.7947     | 0.7970     | ~ neutral |
| Context Recall    | 0.9250     | 0.8917     | ▼ −0.0333 |

**Decision: SKIP enrichment**

Enrichment improved Faithfulness but hurt Recall. RecursiveChar does not track
section metadata (section field is empty for all chunks), so the enrichment header
reduces to `Document: filename / Section: / Type: recursive` — the empty Section
field adds noise to the embedding rather than signal. Enrichment would pay off more
with SemanticBlock, which carries real section names. For RC, the header shifts
retrieval away from some relevant chunks.

---

### Final Results

| Strategy                | Faithfulness | Ans Relevancy | Ctx Precision | Ctx Recall | SUM    |
|-------------------------|-------------|---------------|---------------|------------|--------|
| SlidingWindow           | 0.8838      | 0.8205        | 0.7898        | 0.8750     | 3.3691 |
| SemanticBlock-v3        | 0.8604      | 0.8346        | 0.7510        | 0.8167     | 3.2627 |
| RecursiveChar           | **0.8879**  | 0.8173        | **0.7947**    | **0.9250** | **3.4249** |
| RecursiveChar+Enrichment| 0.9151      | 0.8166        | 0.7970        | 0.8917     | 3.4204 |

**Production pipeline: pymupdf4llm + RecursiveChar (no enrichment)**

---

### What to implement in PDFConnector

Replace the current `SlidingWindowChunker` in `backend/connectors/pdf_connector.py`
with:
1. `pymupdf4llm.to_markdown()` for extraction (already using PyMuPDF, upgrade the call)
2. `RecursiveChar` chunker with `max_tokens=512`, `overlap_tokens=50`

---

### What's next

Context Precision at 0.7947 is the weakest metric — the retriever is fetching
loosely relevant chunks alongside the correct ones. This is where reranking helps.

**Experiment 07** tests Cohere rerank on top of this pipeline:
- Baseline: top-5 cosine similarity (this experiment's winner)
- Condition 2: top-20 cosine, no reranker (isolates candidate pool effect)
- Condition 3: top-20 cosine → Cohere rerank → top-5 (actual reranker contribution)

Expected outcome: Context Precision improves, Faithfulness stays high,
Recall holds because the candidate pool is larger.
